In [21]:
from IPython.display import display, Math
import numpy as np

def array_to_latex(array, real=False, precision=4, array_name=None):
    # Convert input to a numpy array for consistent handling
    arr = np.asarray(array)
    if real:
        arr = arr.real
        
    # Standardize 1D arrays to 2D row vectors by default
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
        
    def format_element(val):
        # Format complex numbers elegantly for LaTeX
        if np.iscomplexobj(val):
            r, i = val.real, val.imag
            if abs(i) < 1e-9:
                return f"{r:.{precision}g}"
            if abs(r) < 1e-9:
                return f"{i:.{precision}g}j"
            sign = "+" if i >= 0 else "-"
            return f"{r:.{precision}g} {sign} {abs(i):.{precision}g}j"
        else:
            # Format integers and floats cleanly to avoid long decimal tails
            if isinstance(val, (int, np.integer)):
                return str(val)
            return f"{val:.{precision}g}"

    # Build the LaTeX matrix body
    rows = []
    for row in arr:
        formatted_elements = [format_element(x) for x in row]
        rows.append(" & ".join(formatted_elements))
        
    # Join rows using the standard LaTeX newline
    matrix_body = r" \\ ".join(rows)
    
    # Display the final LaTeX formula
    if array_name is not None:
        display(Math(rf"{array_name} = \begin{{bmatrix}} {matrix_body} \end{{bmatrix}}"))
    else:
        display(Math(rf"\begin{{bmatrix}} {matrix_body} \end{{bmatrix}}"))

In [9]:
import numpy as np

# Define the 7x7 FMO Hamiltonian matrix (H_exc) in cm^-1
# Values are taken from equation V.1 of the reference
H_exc = np.array([
    [ 200.0, -96.0,   5.0,  -4.4,   4.7, -12.6,  -6.2],
    [-96.0,  320.0,  33.1,   6.8,   4.5,   7.4,  -0.3],
    [  5.0,   33.1,   0.0, -51.1,  38.3,  -8.4,  -0.1],
    [ -4.4,    6.8, -51.1, 110.0, -76.6, -14.2, -67.0],
    [  4.7,    4.5,  38.3, -76.6, 270.0,  78.3,  -0.1],
    [-12.6,    7.4,  -8.4, -14.2,  78.3, 420.0,  38.3],
    [ -6.2,   -0.3,  -0.1, -67.0,  -0.1,  38.3, 230.0]
])

# Diagonalize the Hamiltonian matrix
# eigh is optimized for Hermitian/symmetric matrices
eigenvalues, eigenvectors = np.linalg.eigh(H_exc)

eigenvalues

array([-26.81026845,  65.9232251 , 147.42063749, 243.87569814,
       276.45594373, 373.79891201, 469.33585198])

In [22]:
array_to_latex(eigenvectors)

<IPython.core.display.Math object>

In [11]:
import numpy as np

# Initialize a list to hold the 7 operators S_i in the site basis
S_site = []
for i in range(7):
    # Create a 7x7 matrix of zeros
    op = np.zeros((7, 7))
    # Set the diagonal element corresponding to site i to 1 (|i><i|)
    op[i, i] = 1.0
    S_site.append(op)

# Transform each S_i into the exciton basis
# U is the eigenvectors matrix computed previously
U = eigenvectors
# U_dag is the conjugate transpose of U
U_dag = np.conjugate(U.T)

S_exciton = []
for i in range(7):
    # Apply the change of basis: S_exciton = U_dag * S_site * U
    op_exc = U_dag @ S_site[i] @ U
    S_exciton.append(op_exc)



In [12]:
import numpy as np

# Physical constants for unit conversion
# Energies are in cm^-1, time in fs. We need consistent units.
# 1 cm^-1 = 29979245800 Hz. k_B in cm^-1/K is approx 0.695
K_B = 0.695
TEMPERATURE = 347.0
# Calculate the inverse temperature beta
BETA = 1.0 / (K_B * TEMPERATURE)

# Environment parameters
LAMBDA_REORG = 35.0 # cm^-1
# Omega inverse is 1 fs. We need Omega in cm^-1 for consistency.
# 1 fs = 10^-15 s. In cm^-1, 1 fs^-1 is approx 33356.4 cm^-1
# Note: Ensure all parameters are converted to the same unit system (e.g., cm^-1)
OMEGA = 33356.4 

def spectral_function(omega):
    # Handle the limit for omega -> 0 to avoid division by zero
    if np.abs(omega) < 1e-10:
        # Evaluate the limit of C(omega) as omega goes to 0
        return 4.0 * LAMBDA_REORG * OMEGA * (1.0 / BETA) / (OMEGA**2)
    
    # Calculate the Drude-Lorentz spectral function
    bose_einstein_factor = 1.0 / (1.0 - np.exp(-omega * BETA))
    drude_lorentz_shape = (OMEGA) / (omega**2 + OMEGA**2)
    
    return 4.0 * LAMBDA_REORG * omega * drude_lorentz_shape * bose_einstein_factor

# Calculate the transition frequencies matrix (omega_matrix)
# omega_matrix[alpha, beta] = epsilon_beta - epsilon_alpha
num_states = len(eigenvalues)
omega_matrix = np.zeros((num_states, num_states))
for alpha in range(num_states):
    for beta in range(num_states):
        omega_matrix[alpha, beta] = eigenvalues[beta] - eigenvalues[alpha]

# Evaluate C(omega) for all transitions
# C_matrix[alpha, beta] contains the spectral function evaluated at omega_matrix[alpha, beta]
C_matrix = np.zeros((num_states, num_states))
for alpha in range(num_states):
    for beta in range(num_states):
        C_matrix[alpha, beta] = spectral_function(omega_matrix[alpha, beta])

In [23]:
array_to_latex(omega_matrix)

<IPython.core.display.Math object>

In [24]:
array_to_latex(C_matrix)

<IPython.core.display.Math object>